1- ARIMA

In [ ]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
import warnings

warnings.filterwarnings("ignore")

# ───────── CONFIG ─────────
RUTA_VENTAS     = Path(r"C:\Users\Lenovo\Desktop\LABO 3\sell-in.txt")
RUTA_LISTA_780  = Path(r"C:\Users\Lenovo\Desktop\LABO 3\780_a_predecir.txt")
CARPETA_SALIDA  = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
VENTANAS        = [1, 3, 6, 9]           # ← ventanas a probar
ARIMA_ORDER     = (1, 1, 1)
SEED            = 14
# ──────────────────────────

# 0) Fijo semillas globales
random.seed(SEED)
np.random.seed(SEED)

# 1) Leer histórico de ventas
df = pd.read_csv(RUTA_VENTAS, sep=None, engine="python")

# 2) Leer lista exacta de 780 product_id
with open(RUTA_LISTA_780, 'r', encoding='utf-8') as f:
    product_ids = [
        int(line.strip()) for line in f
        if line.strip() and not line.lower().startswith('product')
    ]

# 3) Preparar campo fecha
df['periodo'] = df['periodo'].astype(str).str.zfill(6)
df['anio']    = df['periodo'].str[:4].astype(int)
df['mes']     = df['periodo'].str[4:6].astype(int)
df['fecha']   = pd.to_datetime(
    df['anio'].astype(str) + df['mes'].astype(str) + '01',
    format='%Y%m%d'
)

# 4) Filtrar los 780 SKU
df = df[df['product_id'].isin(product_ids)]

# 5) Serie mensual por SKU
mensual = (
    df.groupby(['product_id', 'fecha'])['tn']
      .sum()
      .sort_index()
)

# 6) Generar un CSV por cada ventana de respaldo
for ventana in VENTANAS:
    predicciones = []

    for pid in product_ids:
        serie = mensual.loc[pid] if pid in mensual.index.get_level_values(0) else pd.Series(dtype=float)
        serie = serie.asfreq('MS', fill_value=0)

        # fallback si la serie es muy corta o inactiva
        if len(serie) < 4 or serie.sum() == 0:
            y_hat = float(serie.iloc[-1]) if len(serie) > 0 else 0.0
        else:
            try:
                model = ARIMA(serie, order=ARIMA_ORDER).fit()
                y_hat = float(model.forecast(1).iloc[0])
            except Exception:
                y_hat = float(serie.tail(ventana).mean())

        predicciones.append((pid, max(0, round(y_hat, 5))))

    pred = pd.DataFrame(predicciones, columns=['product_id', 'tn'])
    archivo = CARPETA_SALIDA / f"submission_t780_arima_win{ventana}.csv"
    pred.to_csv(archivo, index=False, float_format="%.5f")
    print(f"✅ CSV ventana {ventana} guardado → {archivo.name}")

print("🏁 Listo: se generaron los 4 archivos con ARIMA + fallback dinámico.")  

2- AUTOARIMA

In [14]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
import warnings
from joblib import Parallel, delayed
from pmdarima import auto_arima

warnings.filterwarnings("ignore")

# ───────── CONFIG ─────────
RUTA_VENTAS     = Path(r"C:\Users\Lenovo\Desktop\LABO 3\sell-in.txt")
RUTA_LISTA_780  = Path(r"C:\Users\Lenovo\Desktop\LABO 3\780_a_predecir.txt")
CARPETA_SALIDA  = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
VENTANAS        = [1, 3, 6, 9]           # ventanas a probar para fallback
SEASONAL        = True                  # si hay componente estacional
M               = 12                    # periodicidad (12 meses)
SEED            = 14
# ──────────────────────────

# 0) Fijo semillas globales
random.seed(SEED)
np.random.seed(SEED)

# 1) Cargar ventas y lista de IDs
df = pd.read_csv(RUTA_VENTAS, sep=None, engine="python")
with open(RUTA_LISTA_780, 'r', encoding='utf-8') as f:
    product_ids = [
        int(line.strip())
        for line in f
        if line.strip() and not line.lower().startswith("product")
    ]

mensual = (
    df.assign(periodo=lambda d: d['periodo'].astype(str).str.zfill(6))
      .assign(fecha=lambda d: pd.to_datetime(
          d['periodo'].str[:4] + d['periodo'].str[4:6] + '01',
          format='%Y%m%d'
      ))
      .query("product_id in @product_ids")
      .groupby(['product_id', 'fecha'])['tn']
      .sum()
      .sort_index()
)

def predecir_producto(pid, ventana):
    # Extrae o construye serie vacía
    serie = mensual.loc[pid] if pid in mensual.index.get_level_values(0) else pd.Series(dtype=float)
    serie = serie.asfreq('MS', fill_value=0)

    # Caso pocos datos o serie “muerta”
    if len(serie) < 4 or serie.sum() == 0:
        y_hat = float(serie.iloc[-1]) if len(serie) > 0 else 0.0
    else:
        try:
            # Ajuste automático de ARIMA con semilla
            arima = auto_arima(
                serie,
                seasonal=SEASONAL,
                m=M,
                stepwise=True,
                information_criterion='aic',
                error_action='ignore',
                suppress_warnings=True,
                max_p=3, max_q=3,
                max_P=2, max_Q=2,
                d=None, D=None,
                random_state=SEED
            )
            y_hat = float(arima.predict(n_periods=1)[0])
        except Exception:
            # Fallback: media de últimas `ventana` observaciones
            y_hat = float(serie.tail(ventana).mean())

    return pid, max(0, round(y_hat, 5))


# 6) Generar un CSV por cada ventana, en paralelo
for ventana in VENTANAS:
    resultados = Parallel(n_jobs=-1)(
        delayed(predecir_producto)(pid, ventana) for pid in product_ids
    )
    pred = pd.DataFrame(resultados, columns=['product_id','tn'])
    archivo = CARPETA_SALIDA / f"submission_t780_autoarima_win{ventana}.csv"
    pred.to_csv(archivo, index=False, float_format="%.5f")
    print(f"✅ CSV ventana {ventana} guardado → {archivo.name}")

print("🏁 Listo: se generaron los 4 archivos con auto_arima + fallback dinámico.")  

✅ CSV ventana 1 guardado → submission_t780_autoarima_win1.csv
✅ CSV ventana 3 guardado → submission_t780_autoarima_win3.csv
✅ CSV ventana 6 guardado → submission_t780_autoarima_win6.csv
✅ CSV ventana 9 guardado → submission_t780_autoarima_win9.csv
🏁 Listo: se generaron los 4 archivos con auto_arima + fallback dinámico.


3- LIGHT GBM

In [15]:
import pandas as pd
import numpy as np
import random
import lightgbm as lgb
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
import warnings

warnings.filterwarnings("ignore")

# ───────── CONFIG ─────────
RUTA_VENTAS     = Path(r"C:\Users\Lenovo\Desktop\LABO 3\sell-in.txt")
RUTA_LISTA_780  = Path(r"C:\Users\Lenovo\Desktop\LABO 3\780_a_predecir.txt")
CARPETA_SALIDA  = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
MAX_LAG         = 12       # número de lags
ROLL_WINDOWS    = [3, 6]   # ventanas para medias móviles
SEED            = 14       # semilla para reproducibilidad
# ──────────────────────────

# 0) Fijo semillas globales
random.seed(SEED)
np.random.seed(SEED)

# 1) Cargo datos
df = pd.read_csv(RUTA_VENTAS, sep=None, engine="python")
with open(RUTA_LISTA_780, 'r', encoding='utf-8') as f:
    product_ids = [int(l.strip()) for l in f
                   if l.strip() and not l.lower().startswith('product')]

# 2) Preparo fecha y filtro
df['periodo'] = df['periodo'].astype(str).str.zfill(6)
df['fecha']   = pd.to_datetime(
    df['periodo'].str[:4] + df['periodo'].str[4:6] + '01',
    format='%Y%m%d'
)
df = df[df['product_id'].isin(product_ids)]

# 3) Serie mensual
mensual = (
    df.groupby(['product_id','fecha'])['tn']
      .sum()
      .sort_index()
      .unstack(level=0)
      .fillna(0)
)

# 4) Construcción de dataset de entrenamiento
rows = []
for pid in mensual.columns:
    serie = mensual[pid]
    for idx in range(MAX_LAG, len(serie)):
        feats = {f'lag_{l}': serie.iloc[idx-l] for l in range(1, MAX_LAG+1)}
        for w in ROLL_WINDOWS:
            feats[f'roll_mean_{w}'] = serie.iloc[idx-w:idx].mean()
        feats['product_id'] = pid
        feats['target']     = serie.iloc[idx]
        rows.append(feats)

train_df = pd.DataFrame(rows)

# 5) Preparo LightGBM
train_df['product_id'] = train_df['product_id'].astype('category')
FEATURES = [c for c in train_df.columns if c not in ['target']]
dtrain = lgb.Dataset(
    train_df[FEATURES],
    label=train_df['target'],
    categorical_feature=['product_id'],
    free_raw_data=False
)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'seed': SEED,
    'verbose': -1
}

model = lgb.train(
    params,
    dtrain,
    num_boost_round=500
)

# 6) Preparo features para el mes siguiente
last_date = mensual.index.max()
pred_date = last_date + MonthBegin()

pred_rows = []
for pid in product_ids:
    serie = mensual.get(pid, pd.Series(dtype=float))
    if serie.empty:
        serie = pd.Series(0, index=mensual.index)

    feats = {'product_id': pid}
    for l in range(1, MAX_LAG+1):
        feats[f'lag_{l}'] = serie.iloc[-l] if len(serie) >= l else 0.0
    for w in ROLL_WINDOWS:
        vals = serie.iloc[-w:] if len(serie) >= w else serie
        feats[f'roll_mean_{w}'] = vals.mean() if not vals.empty else 0.0

    pred_rows.append(feats)

pred_df = pd.DataFrame(pred_rows)
pred_df['product_id'] = pred_df['product_id'].astype('category')

# 7) Predicción
y_pred = model.predict(pred_df[FEATURES])
y_pred = np.maximum(0, np.round(y_pred, 5))

# 8) Exportar submission
submission = pd.DataFrame({
    'product_id': product_ids,
    'tn':         y_pred
})
out_csv = CARPETA_SALIDA / "submission_t780_lgbm.csv"
submission.to_csv(out_csv, index=False, float_format="%.5f")

print(f"✅ Submission LightGBM guardado → {out_csv.name}")

✅ Submission LightGBM guardado → submission_t780_lgbm.csv


4- LIGHT GBM mejorado

In [16]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import warnings

warnings.filterwarnings("ignore")

# ───────── CONFIG ─────────
RUTA_VENTAS     = Path(r"C:\Users\Lenovo\Desktop\LABO 3\sell-in.txt")
RUTA_LISTA_780  = Path(r"C:\Users\Lenovo\Desktop\LABO 3\780_a_predecir.txt")
CARPETA_SALIDA  = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
MAX_LAG         = 12         # últimos 12 meses de lag
ROLL_WINDOWS    = [3, 6, 12] # medias y std sobre ventanas de 3, 6 y 12 meses
VAL_MONTHS      = 12         # usar últimos 12 meses para validación
HALF_LIFE       = 6          # meses de decaimiento exponencial en sample weights
SEED            = 14         # semilla para reproducibilidad
# ──────────────────────────

# 0) Fijo semillas globales
random.seed(SEED)
np.random.seed(SEED)

# 1) Cargo datos y filtro de productos
df = pd.read_csv(RUTA_VENTAS, sep=None, engine="python")
with open(RUTA_LISTA_780, 'r', encoding='utf-8') as f:
    product_ids = [
        int(line.strip())
        for line in f
        if line.strip() and not line.lower().startswith('product')
    ]

# Preparo la fecha
df['periodo'] = df['periodo'].astype(str).str.zfill(6)
df['fecha']   = pd.to_datetime(
    df['periodo'].str[:4] + df['periodo'].str[4:6] + '01',
    format='%Y%m%d'
)
df = df[df['product_id'].isin(product_ids)]

# 2) Pivot mensual: índice = fecha, columnas = product_id
mensual = (
    df.groupby(['product_id', 'fecha'])['tn']
      .sum()
      .unstack(level=0)
      .sort_index()
      .fillna(0)
)

# 3) Construcción de features
records = []
for pid in mensual.columns:
    serie = mensual[pid]
    for idx in range(MAX_LAG, len(serie)):
        fecha_i = serie.index[idx]
        feats = {f'lag_{l}': serie.iloc[idx - l] for l in range(1, MAX_LAG + 1)}
        for w in ROLL_WINDOWS:
            window = serie.iloc[idx - w:idx]
            feats[f'roll_mean_{w}'] = window.mean()
            feats[f'roll_std_{w}']  = window.std()
        feats['diff_1_2']   = serie.iloc[idx - 1] - serie.iloc[idx - 2]
        feats['ratio_1_12'] = serie.iloc[idx - 1] / (serie.iloc[idx - 12] + 1e-6)
        feats['month']   = fecha_i.month
        feats['quarter'] = fecha_i.quarter
        feats['year']    = fecha_i.year
        feats['product_id'] = pid
        feats['fecha']      = fecha_i
        feats['target']     = serie.iloc[idx]
        records.append(feats)

full_df = pd.DataFrame(records)

# 4) Split train / valid según fecha
max_date = mensual.index.max()
cut_date = max_date - pd.DateOffset(months=VAL_MONTHS)
train_df = full_df[full_df['fecha'] <= cut_date].copy()
valid_df = full_df[full_df['fecha'] >  cut_date].copy()

# 5) Sample weights exponenciales en entrenamiento
train_df['age'] = (
    (max_date.year - train_df['fecha'].dt.year) * 12 +
    (max_date.month - train_df['fecha'].dt.month)
)
train_df['weight'] = np.exp(- train_df['age'] / HALF_LIFE)
valid_df['weight'] = 1.0

# 6) Preparar matrices para LightGBM
cat_feats = ['product_id', 'month', 'quarter', 'year']
for df_ in (train_df, valid_df):
    for c in cat_feats:
        df_[c] = df_[c].astype('category')

FEATURES = [
    c for c in train_df.columns
    if c not in ['target','weight','fecha','age']
]

X_train = train_df[FEATURES]
y_train = train_df['target']
w_train = train_df['weight']

X_val = valid_df[FEATURES]
y_val = valid_df['target']
w_val = valid_df['weight']

# 7) Definir y entrenar LGBMRegressor con seed y callbacks
model = LGBMRegressor(
    objective='regression',
    metric='rmse',
    learning_rate=0.01,
    num_leaves=64,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_estimators=10000,
    random_state=SEED,    # semilla fija
    verbose=-1
)

model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_sample_weight=[w_train, w_val],
    eval_names=['train','valid'],
    eval_metric='rmse',
    categorical_feature=cat_feats,
    callbacks=[
        early_stopping(stopping_rounds=100),
        log_evaluation(period=100)
    ]
)

# 8) Preparar features para predecir el próximo mes
next_date = max_date + MonthBegin()
pred_rows = []
for pid in product_ids:
    serie = mensual.get(pid, pd.Series(0, index=mensual.index))
    feats = {f'lag_{l}': serie.iloc[-l] if len(serie) >= l else 0.0
             for l in range(1, MAX_LAG + 1)}
    for w in ROLL_WINDOWS:
        window = serie.iloc[-w:] if len(serie) >= w else serie
        feats[f'roll_mean_{w}'] = window.mean()
        feats[f'roll_std_{w}']  = window.std()
    feats['diff_1_2']   = (serie.iloc[-1] - serie.iloc[-2]) if len(serie) >= 2 else 0.0
    feats['ratio_1_12'] = (serie.iloc[-1] / (serie.iloc[-12] + 1e-6)) if len(serie) >= 12 else 0.0
    feats['month']   = next_date.month
    feats['quarter'] = next_date.quarter
    feats['year']    = next_date.year
    feats['product_id'] = pid
    pred_rows.append(feats)

pred_df = pd.DataFrame(pred_rows)
for c in cat_feats:
    pred_df[c] = pred_df[c].astype('category')

# 9) Predicción y guardado de submission
y_pred = model.predict(pred_df[FEATURES])
y_pred = np.maximum(0, np.round(y_pred, 5))

submission = pd.DataFrame({
    'product_id': product_ids,
    'tn':         y_pred
})
out_csv = CARPETA_SALIDA / "submission_t780_lgbm_enhanced.csv"
submission.to_csv(out_csv, index=False, float_format="%.5f")

print(f"✅ Submission mejorada guardada → {out_csv.name}")

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 49.7201	valid's rmse: 48.8302
[200]	train's rmse: 30.0131	valid's rmse: 33.1765
[300]	train's rmse: 23.8681	valid's rmse: 30.2764
[400]	train's rmse: 21.2995	valid's rmse: 30.2891
Early stopping, best iteration is:
[331]	train's rmse: 22.8649	valid's rmse: 30.1887
✅ Submission mejorada guardada → submission_t780_lgbm_enhanced.csv


5- LIGHT GBM variante

In [17]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import warnings

warnings.filterwarnings("ignore")

# ───────── RUTAS (IGUALES) ─────────
BASE_DIR        = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
RUTA_VENTAS     = BASE_DIR / "sell-in.txt"
RUTA_LISTA_780  = BASE_DIR / "780_a_predecir.txt"
RUTA_CATALOGO   = BASE_DIR / "tb_productos.txt"      # ← nuevo
CARPETA_SALIDA  = BASE_DIR
# ───────────────────────────────────

SEED = 14
random.seed(SEED)
np.random.seed(SEED)

MAX_LAG      = 12
ROLL_WINDOWS = [3, 6, 12]
VAL_MONTHS   = 12
HALF_LIFE    = 6

# 1) ----------------- leer datasets -----------------
df = pd.read_csv(RUTA_VENTAS, sep=None, engine="python")

with open(RUTA_LISTA_780, 'r', encoding='utf-8') as f:
    product_ids = [
        int(l.strip())
        for l in f
        if l.strip() and not l.lower().startswith('product')
    ]

cat_df = pd.read_csv(RUTA_CATALOGO, sep=None, engine="python")
cat_df = cat_df[cat_df['product_id'].isin(product_ids)]

# columnas extra que quieras usar como categóricas
extra_cat_cols = [c for c in cat_df.columns if c != 'product_id'][:3]

# 2) -------------- preparar fecha -------------------
df['periodo'] = df['periodo'].astype(str).str.zfill(6)
df['fecha']   = pd.to_datetime(df['periodo'].str[:4] + df['periodo'].str[4:6] + '01',
                               format='%Y%m%d')
df = df[df['product_id'].isin(product_ids)]

# 3) -------------- tabla mensual --------------------
mensual = (df.groupby(['product_id', 'fecha'])['tn']
             .sum()
             .unstack(level=0)
             .sort_index()
             .fillna(0))

# 4) ---- features por producto/mes (+ catálogo) -----
records = []
for pid in mensual.columns:
    serie = mensual[pid]
    meta  = cat_df[cat_df['product_id']==pid].iloc[0] if pid in cat_df['product_id'].values else None
    for idx in range(MAX_LAG, len(serie)):
        fecha_i = serie.index[idx]
        feats = {f'lag_{l}': serie.iloc[idx-l] for l in range(1, MAX_LAG+1)}
        for w in ROLL_WINDOWS:
            window = serie.iloc[idx-w:idx]
            feats[f'roll_mean_{w}'] = window.mean()
            feats[f'roll_std_{w}']  = window.std()
        feats['month']   = fecha_i.month
        feats['quarter'] = fecha_i.quarter
        feats['year']    = fecha_i.year
        feats['product_id'] = pid
        feats['fecha']      = fecha_i
        feats['target']     = serie.iloc[idx]
        if meta is not None:
            for col in extra_cat_cols:
                feats[col] = meta[col]
        records.append(feats)

full_df = pd.DataFrame(records)

# 5) --------- split temporal train / valid ----------
max_date = mensual.index.max()
cut_date = max_date - pd.DateOffset(months=VAL_MONTHS)
train_df = full_df[full_df['fecha'] <= cut_date].copy()
valid_df = full_df[full_df['fecha'] >  cut_date].copy()

# 6) --------- sample weights (exponencial) ----------
for d in (train_df, valid_df):
    d['age'] = (max_date.year - d['fecha'].dt.year) * 12 + (max_date.month - d['fecha'].dt.month)
train_df['weight'] = np.exp(-train_df['age'] / HALF_LIFE)
valid_df['weight'] = 1.0

# 7) ---------- log-transform objetivo ---------------
train_df['target_log'] = np.log1p(train_df['target'])
valid_df['target_log'] = np.log1p(valid_df['target'])

# 8) ---------- preparar matrices LGBM ---------------
cat_feats = ['product_id', 'month', 'quarter', 'year'] + extra_cat_cols
for d in (train_df, valid_df):
    for c in cat_feats:
        d[c] = d[c].astype('category')

FEATURES = [
    c for c in train_df.columns
    if c not in ['target','target_log','weight','fecha','age']
]

X_train, y_train, w_train = train_df[FEATURES], train_df['target_log'], train_df['weight']
X_val,   y_val,   w_val   = valid_df[FEATURES], valid_df['target_log'], valid_df['weight']

# 9) -------- definir y entrenar LGBMRegressor con seed --------
model = LGBMRegressor(
    objective='poisson',
    learning_rate=0.05,
    num_leaves=64,
    n_estimators=8000,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    max_depth=-1,
    random_state=SEED,    # ← semilla para reproducibilidad
    verbose=-1
)

model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    eval_sample_weight=[w_val],
    eval_metric='rmse',
    categorical_feature=cat_feats,
    callbacks=[early_stopping(stopping_rounds=200), log_evaluation(period=200)]
)

# 10) -------- preparar features del próximo mes -------
next_date = max_date + MonthBegin()
pred_rows = []
for pid in product_ids:
    serie = mensual.get(pid, pd.Series(0, index=mensual.index))
    meta  = cat_df[cat_df['product_id']==pid].iloc[0] if pid in cat_df['product_id'].values else None

    feats = {f'lag_{l}': serie.iloc[-l] if len(serie)>=l else 0.0 for l in range(1, MAX_LAG+1)}
    for w in ROLL_WINDOWS:
        window = serie.iloc[-w:] if len(serie)>=w else serie
        feats[f'roll_mean_{w}'] = window.mean()
        feats[f'roll_std_{w}']  = window.std()
    feats['month']   = next_date.month
    feats['quarter'] = next_date.quarter
    feats['year']    = next_date.year
    feats['product_id'] = pid
    if meta is not None:
        for col in extra_cat_cols:
            feats[col] = meta[col]
    pred_rows.append(feats)

pred_df = pd.DataFrame(pred_rows)
for c in cat_feats:
    pred_df[c] = pred_df[c].astype('category')

# 11) ------ predicción & CSV ------
y_pred_log = model.predict(pred_df[FEATURES])
y_pred     = np.expm1(y_pred_log)                # deshacer log1p
y_pred     = np.maximum(0, np.round(y_pred, 5))

submission = pd.DataFrame({'product_id': product_ids, 'tn': y_pred})
out_csv    = CARPETA_SALIDA / "submission_t780_lgbm_catalogo.csv"
submission.to_csv(out_csv, index=False, float_format="%.5f")

print(f"✅ Submission LGBM+catálogo guardada → {out_csv.name}")

Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.416299	valid_0's poisson: -0.145928
Early stopping, best iteration is:
[118]	valid_0's rmse: 0.421971	valid_0's poisson: -0.156796
✅ Submission LGBM+catálogo guardada → submission_t780_lgbm_catalogo.csv


In [18]:
df = pd.read_csv(
    "C:/Users/Lenovo/Desktop/LABO 3/sell-in.txt",
    sep="\t",          # <— aquí le decimos que es TSV
    engine="python"
)
print(df.columns.tolist())

['periodo', 'customer_id', 'product_id', 'plan_precios_cuidados', 'cust_request_qty', 'cust_request_tn', 'tn']


In [19]:
# A partir de aquí tu pipeline ya podrá hacer:
df['periodo'] = df['periodo'].astype(str).str.zfill(6)
df['fecha']   = pd.to_datetime(
    df['periodo'].str[:4] + df['periodo'].str[4:6] + '01',
    format='%Y%m%d'
)

6- LIGHT GBM Optuna

In [20]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor, early_stopping
import optuna
from optuna.samplers import TPESampler
import warnings

warnings.filterwarnings("ignore")

# ────────── RUTAS ──────────
BASE     = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
VENTAS   = BASE / "sell-in.txt"
LISTA    = BASE / "780_a_predecir.txt"
OUT_SUB  = BASE / "submission_t780_lgbm_optuna.csv"
# ──────────────────────────

SEED = 14
random.seed(SEED)
np.random.seed(SEED)

# 1) Leer ventas como TSV y limpiar columnas
df = pd.read_csv(VENTAS, sep="\t", engine="python")
df.columns = df.columns.str.strip()

# 2) Detectar columnas 'periodo' y 'product_id'
period_col = next((c for c in df.columns if "period" in c.lower()), None)
prod_col   = next((c for c in df.columns if c.lower()=="product_id"), None)
if period_col is None or prod_col is None:
    raise KeyError(f"Columnas encontradas: {df.columns.tolist()}")

df = df.rename(columns={period_col:"periodo", prod_col:"product_id"})

# 3) Leer lista de 780 SKUs
with open(LISTA, "r", encoding="utf-8") as f:
    prods = [int(l) for l in f if l.strip() and not l.lower().startswith("product")]

# 4) Formatear fecha y filtrar
df["periodo"] = df["periodo"].astype(str).str.zfill(6)
df["fecha"] = pd.to_datetime(
    df["periodo"].str[:4] + df["periodo"].str[4:6] + "01",
    format="%Y%m%d",
    errors="coerce"
)
df = df[df["product_id"].isin(prods) & df["fecha"].notna()]

# 5) Serie mensual de toneladas
mensual = (
    df.groupby(["product_id","fecha"])["tn"]
      .sum()
      .unstack(0)
      .sort_index()
      .asfreq("MS", fill_value=0)
)

# 6) Variables globales
gmean = mensual.mean(axis=1)
gstd  = mensual.std(axis=1)

# 7) Feature engineering
MAX_LAG = 12
ROLL    = [3, 6, 12]
records = []
for pid in mensual.columns:
    s = mensual[pid]
    for i in range(MAX_LAG, len(s)):
        dt = s.index[i]
        r = {"product_id": pid, "fecha": dt, "target": s.iloc[i]}
        # lags
        for lag in range(1, MAX_LAG+1):
            r[f"lag_{lag}"] = s.iloc[i-lag]
        # rolling stats + min/max
        for w in ROLL:
            win = s.iloc[i-w:i]
            r[f"mean_{w}"] = win.mean()
            r[f"std_{w}"]  = win.std()
            r[f"min_{w}"]  = win.min()
            r[f"max_{w}"]  = win.max()
        # fecha cíclica
        m = dt.month - 1
        r["sin_m"] = np.sin(2*np.pi*m/12)
        r["cos_m"] = np.cos(2*np.pi*m/12)
        # tendencia global
        r["gmean"] = gmean.loc[dt]
        r["gstd"]  = gstd.loc[dt]
        records.append(r)

df_feat = pd.DataFrame(records).fillna(0)

# 8) Split temporal
VAL_MONTHS = 12
cut = mensual.index.max() - pd.DateOffset(months=VAL_MONTHS)
train = df_feat[df_feat["fecha"] <= cut].copy()
valid = df_feat[df_feat["fecha"] >  cut].copy()

X_tr = train.drop(["target","fecha"], axis=1)
y_tr = train["target"]
X_va = valid.drop(["target","fecha"], axis=1)
y_va = valid["target"]

FEATURES = X_tr.columns.tolist()

# 9) Optuna con semilla para LGBM + early stopping
def objective(trial):
    params = {
        "objective":        "regression",
        "metric":           "rmse",
        "learning_rate":    trial.suggest_loguniform("lr", 1e-3, 1e-1),
        "num_leaves":       trial.suggest_int("leaves", 31, 128),
        "max_depth":        trial.suggest_int("depth", 6, 12),
        "feature_fraction": trial.suggest_uniform("ff", 0.6, 1.0),
        "bagging_fraction": trial.suggest_uniform("bf", 0.6, 1.0),
        "bagging_freq":     trial.suggest_int("bfr", 1, 10),
        "lambda_l1":        trial.suggest_loguniform("l1", 1e-3, 1.0),
        "lambda_l2":        trial.suggest_loguniform("l2", 1e-3, 1.0),
        "random_state":     SEED,
        "n_jobs":           -1
    }
    m = LGBMRegressor(**params, n_estimators=10000)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="rmse",
        callbacks=[early_stopping(stopping_rounds=100)]
    )
    pred = m.predict(X_va)
    return mean_squared_error(y_va, pred, squared=False)

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=30)
best = study.best_params

# 10) Entrenar final y generar submission
model = LGBMRegressor(**best, n_estimators=10000, random_state=SEED, n_jobs=-1)
model.fit(pd.concat([X_tr, X_va]), pd.concat([y_tr, y_va]))

# 11) Preparar features para el próximo mes
next_dt    = mensual.index.max() + MonthBegin()
last_gmean = gmean.iloc[-1]
last_gstd  = gstd.iloc[-1]

preds = []
for pid in mensual.columns:
    s = mensual[pid]
    r = {"product_id": pid}
    for lag in range(1, MAX_LAG+1):
        r[f"lag_{lag}"] = s.iloc[-lag]
    for w in ROLL:
        win = s.iloc[-w:]
        r[f"mean_{w}"] = win.mean()
        r[f"std_{w}"]  = win.std()
        r[f"min_{w}"]  = win.min()
        r[f"max_{w}"]  = win.max()
    m = next_dt.month - 1
    r["sin_m"] = np.sin(2*np.pi*m/12)
    r["cos_m"] = np.cos(2*np.pi*m/12)
    r["gmean"] = last_gmean
    r["gstd"]  = last_gstd
    preds.append(r)

X_pred = pd.DataFrame(preds)[FEATURES]
y_pred = model.predict(X_pred)
y_pred = np.maximum(0, np.round(y_pred, 5))

submission = pd.DataFrame({
    "product_id": mensual.columns,
    "tn":          y_pred
})
submission.to_csv(OUT_SUB, index=False, float_format="%.5f")
print("✅ Submission lista →", OUT_SUB.name)

[I 2025-07-06 21:38:19,448] A new study created in memory with name: no-name-61867852-ce25-4c64-933a-873fc360f0b9


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[568]	valid_0's rmse: 30.2143


[I 2025-07-06 21:38:27,258] Trial 0 finished with value: 30.21426816488993 and parameters: {'lr': 0.010663178702639572, 'leaves': 106, 'depth': 12, 'ff': 0.6032187794119271, 'bf': 0.7238943702020817, 'bfr': 10, 'l1': 0.03462183934556516, 'l2': 0.009012665953349538}. Best is trial 0 with value: 30.21426816488993.


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[370]	valid_0's rmse: 30.7073


[I 2025-07-06 21:38:31,535] Trial 1 finished with value: 30.70728627188688 and parameters: {'lr': 0.011978429326487499, 'leaves': 52, 'depth': 11, 'ff': 0.7369018502458737, 'bf': 0.8155555396268579, 'bfr': 1, 'l1': 0.10458211838885734, 'l2': 0.004266510183230375}. Best is trial 0 with value: 30.21426816488993.


...

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[765]	valid_0's rmse: 30.4685


[I 2025-07-06 21:42:54,090] Trial 28 finished with value: 30.4684716726339 and parameters: {'lr': 0.007811489201454967, 'leaves': 114, 'depth': 11, 'ff': 0.7609651295846013, 'bf': 0.6904673676976345, 'bfr': 9, 'l1': 0.003978444412321561, 'l2': 0.3760605707380777}. Best is trial 18 with value: 30.025915073213458.


Training until validation scores don't improve for 100 rounds


[I 2025-07-06 21:42:56,126] Trial 29 finished with value: 30.329919495966504 and parameters: {'lr': 0.048801477988146665, 'leaves': 109, 'depth': 12, 'ff': 0.622505703387589, 'bf': 0.7372336615266966, 'bfr': 10, 'l1': 0.05136555063932696, 'l2': 0.01389400800815819}. Best is trial 18 with value: 30.025915073213458.


Early stopping, best iteration is:
[160]	valid_0's rmse: 30.3299
✅ Submission lista → submission_t780_lgbm_optuna.csv


7- XG BOOST

In [21]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
import warnings

warnings.filterwarnings("ignore")

# ────────── RUTAS ──────────
BASE     = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
VENTAS   = BASE / "sell-in.txt"
LISTA    = BASE / "780_a_predecir.txt"
OUT_SUB  = BASE / "submission_t780_xgb_optuna.csv"
# ──────────────────────────

SEED = 14
random.seed(SEED)
np.random.seed(SEED)

# 1) Leer ventas como TSV y limpiar columnas
df = pd.read_csv(VENTAS, sep="\t", engine="python")
df.columns = df.columns.str.strip()

# 2) Detectar columnas 'periodo' y 'product_id'
period_col = next(c for c in df.columns if "period" in c.lower())
prod_col   = next(c for c in df.columns if c.lower() == "product_id")
df = df.rename(columns={period_col: "periodo", prod_col: "product_id"})

# 3) Leer lista de 780 SKUs
with open(LISTA, "r", encoding="utf-8") as f:
    prods = [int(l) for l in f if l.strip() and not l.lower().startswith("product")]

# 4) Formatear fecha y filtrar
df["periodo"] = df["periodo"].astype(str).str.zfill(6)
df["fecha"]   = pd.to_datetime(
    df["periodo"].str[:4] + df["periodo"].str[4:6] + "01",
    format="%Y%m%d",
    errors="coerce"
)
df = df[df["product_id"].isin(prods) & df["fecha"].notna()]

# 5) Serie mensual de toneladas
mensual = (
    df.groupby(["product_id","fecha"])["tn"]
      .sum()
      .unstack(0)
      .sort_index()
      .asfreq("MS", fill_value=0)
)

# 6) Variables globales
gmean = mensual.mean(axis=1)
gstd  = mensual.std(axis=1)

# 7) Feature engineering
MAX_LAG = 12
ROLL    = [3, 6, 12]
records = []
for pid in mensual.columns:
    s = mensual[pid]
    for i in range(MAX_LAG, len(s)):
        dt = s.index[i]
        r = {"product_id": pid, "fecha": dt, "target": s.iloc[i]}
        # lags
        for lag in range(1, MAX_LAG+1):
            r[f"lag_{lag}"] = s.iloc[i-lag]
        # rolling stats + min/max
        for w in ROLL:
            win = s.iloc[i-w:i]
            r[f"mean_{w}"] = win.mean()
            r[f"std_{w}"]  = win.std()
            r[f"min_{w}"]  = win.min()
            r[f"max_{w}"]  = win.max()
        # fecha cíclica
        m = dt.month - 1
        r["sin_m"] = np.sin(2*np.pi*m/12)
        r["cos_m"] = np.cos(2*np.pi*m/12)
        # tendencia global
        r["gmean"] = gmean.loc[dt]
        r["gstd"]  = gstd.loc[dt]
        records.append(r)

df_feat = pd.DataFrame(records).fillna(0)

# 8) Split temporal
VAL_MONTHS = 12
cut = mensual.index.max() - pd.DateOffset(months=VAL_MONTHS)
train = df_feat[df_feat["fecha"] <= cut].copy()
valid = df_feat[df_feat["fecha"] >  cut].copy()

X_tr = train.drop(["target","fecha"], axis=1)
y_tr = train["target"]
X_va = valid.drop(["target","fecha"], axis=1)
y_va = valid["target"]

FEATURES = X_tr.columns.tolist()

# 9) Construir DMatrix
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dvalid = xgb.DMatrix(X_va, label=y_va)

# 10) Optuna para XGBoost con early stopping & semilla
def objective(trial):
    params = {
        "tree_method":      "hist",
        "objective":        "reg:squarederror",
        "eval_metric":      "rmse",
        "learning_rate":    trial.suggest_loguniform("eta", 1e-3, 1e-1),
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "subsample":        trial.suggest_uniform("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_uniform("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":        trial.suggest_loguniform("alpha", 1e-3, 1.0),
        "reg_lambda":       trial.suggest_loguniform("lambda", 1e-3, 1.0),
        "seed":             SEED,
        "n_jobs":           -1
    }
    evals = [(dtrain, "train"), (dvalid, "valid")]
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=10000,
        evals=evals,
        early_stopping_rounds=100,
        verbose_eval=False
    )
    preds = bst.predict(dvalid, iteration_range=(0, bst.best_iteration))
    rmse  = mean_squared_error(y_va, preds, squared=False)
    trial.set_user_attr("best_iteration", bst.best_iteration)
    return rmse

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=30)
best_params    = study.best_params
best_iteration = study.best_trial.user_attrs["best_iteration"]

# 11) Entrenar modelo final en todos los datos
dall = xgb.DMatrix(pd.concat([X_tr, X_va]), label=pd.concat([y_tr, y_va]))
final_bst = xgb.train(
    best_params,
    dall,
    num_boost_round=best_iteration
)

# 12) Preparar features para el próximo mes
next_dt    = mensual.index.max() + MonthBegin()
last_gmean = gmean.iloc[-1]
last_gstd  = gstd.iloc[-1]

preds = []
for pid in mensual.columns:
    s = mensual[pid]
    r = {"product_id": pid}
    # lags y rolling
    for lag in range(1, MAX_LAG+1):
        r[f"lag_{lag}"] = s.iloc[-lag]
    for w in ROLL:
        win = s.iloc[-w:]
        r[f"mean_{w}"] = win.mean()
        r[f"std_{w}"]  = win.std()
        r[f"min_{w}"]  = win.min()
        r[f"max_{w}"]  = win.max()
    # fecha cíclica y tendencia global
    m = next_dt.month - 1
    r["sin_m"] = np.sin(2*np.pi*m/12)
    r["cos_m"] = np.cos(2*np.pi*m/12)
    r["gmean"] = last_gmean
    r["gstd"]  = last_gstd
    preds.append(r)

X_pred = pd.DataFrame(preds)[FEATURES]
dpred  = xgb.DMatrix(X_pred)
y_pred = final_bst.predict(dpred, iteration_range=(0, best_iteration))
y_pred = np.maximum(0, np.round(y_pred, 5))

# 13) Guardar submission
submission = pd.DataFrame({
    "product_id": mensual.columns,
    "tn":          y_pred
})
submission.to_csv(OUT_SUB, index=False, float_format="%.5f")
print("✅ Submission XGBoost guardada →", OUT_SUB.name)

[I 2025-07-06 21:46:06,127] A new study created in memory with name: no-name-b6fa219d-7b09-4b02-9dbd-6626ee41667b
[I 2025-07-06 21:46:32,078] Trial 0 finished with value: 30.628252891834464 and parameters: {'eta': 0.010663178702639572, 'max_depth': 10, 'subsample': 0.9352138428624064, 'colsample_bytree': 0.504023474264909, 'min_child_weight': 4, 'alpha': 0.7461243578022466, 'lambda': 0.03462183934556516}. Best is trial 0 with value: 30.628252891834464.
[I 2025-07-06 21:48:03,878] Trial 1 finished with value: 30.617341090848612 and parameters: {'eta': 0.004330807196196427, 'max_depth': 8, 'subsample': 0.61062747121388, 'colsample_bytree': 0.903240678962135, 'min_child_weight': 4, 'alpha': 0.04136819264436807, 'lambda': 0.0010414090720927988}. Best is trial 1 with value: 30.617341090848612.
[I 2025-07-06 21:48:06,803] Trial 2 finished with value: 31.084755572903862 and parameters: {'eta': 0.022197545576322492, 'max_depth': 5, 'subsample': 0.966278796439356, 'colsample_bytree': 0.68712237

✅ Submission XGBoost guardada → submission_t780_xgb_optuna.csv


8- RANDOM FOREST

In [22]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import optuna
from optuna.samplers import TPESampler
import warnings

warnings.filterwarnings("ignore")

# ────────── RUTAS ──────────
BASE     = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
VENTAS   = BASE / "sell-in.txt"
LISTA    = BASE / "780_a_predecir.txt"
OUT_SUB  = BASE / "submission_t780_rf_optuna.csv"
# ──────────────────────────

# 0) Fijo semillas
SEED = 14
random.seed(SEED)
np.random.seed(SEED)

# 1) Leer ventas como TSV y limpiar columnas
df = pd.read_csv(VENTAS, sep="\t", engine="python")
df.columns = df.columns.str.strip()

# 2) Detectar columnas 'periodo' y 'product_id'
period_col = next(c for c in df.columns if "period" in c.lower())
prod_col   = next(c for c in df.columns if c.lower()=="product_id")
df = df.rename(columns={period_col:"periodo", prod_col:"product_id"})

# 3) Leer lista de 780 SKUs
with open(LISTA, "r", encoding="utf-8") as f:
    prods = [int(l) for l in f if l.strip() and not l.lower().startswith("product")]

# 4) Formatear fecha y filtrar
df["periodo"] = df["periodo"].astype(str).str.zfill(6)
df["fecha"]   = pd.to_datetime(
    df["periodo"].str[:4] + df["periodo"].str[4:6] + "01",
    format="%Y%m%d", errors="coerce"
)
df = df[df["product_id"].isin(prods) & df["fecha"].notna()]

# 5) Serie mensual de toneladas
mensual = (
    df.groupby(["product_id","fecha"])["tn"]
      .sum()
      .unstack(0)
      .sort_index()
      .asfreq("MS", fill_value=0)
)

# 6) Variables globales
gmean = mensual.mean(axis=1)
gstd  = mensual.std(axis=1)

# 7) Feature engineering
MAX_LAG = 12
ROLL    = [3, 6, 12]
records = []
for pid in mensual.columns:
    s = mensual[pid]
    for i in range(MAX_LAG, len(s)):
        dt = s.index[i]
        r = {"product_id": pid, "fecha": dt, "target": s.iloc[i]}
        # lags
        for lag in range(1, MAX_LAG+1):
            r[f"lag_{lag}"] = s.iloc[i-lag]
        # rolling stats + min/max
        for w in ROLL:
            win = s.iloc[i-w:i]
            r[f"mean_{w}"] = win.mean()
            r[f"std_{w}"]  = win.std()
            r[f"min_{w}"]  = win.min()
            r[f"max_{w}"]  = win.max()
        # fecha cíclica
        m = dt.month - 1
        r["sin_m"] = np.sin(2*np.pi*m/12)
        r["cos_m"] = np.cos(2*np.pi*m/12)
        # tendencia global
        r["gmean"] = gmean.loc[dt]
        r["gstd"]  = gstd.loc[dt]
        records.append(r)

df_feat = pd.DataFrame(records).fillna(0)

# 8) Split temporal
VAL_MONTHS = 12
cut = mensual.index.max() - pd.DateOffset(months=VAL_MONTHS)
train = df_feat[df_feat["fecha"] <= cut].copy()
valid = df_feat[df_feat["fecha"] >  cut].copy()

X_tr = train.drop(["target","fecha"], axis=1)
y_tr = train["target"]
X_va = valid.drop(["target","fecha"], axis=1)
y_va = valid["target"]

FEATURES = X_tr.columns.tolist()

# 9) Optuna para RandomForest con semilla en el sampler
def objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth":        trial.suggest_int("max_depth", 5, 20),
        "min_samples_split":trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features":     trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state":     SEED,
        "n_jobs":           -1
    }
    rf = RandomForestRegressor(**params)
    rf.fit(X_tr, y_tr)
    pred = rf.predict(X_va)
    return mean_squared_error(y_va, pred, squared=False)

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=30)
best_params = study.best_params

# 10) Entrenar modelo final reproducible
final_rf = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=-1)
final_rf.fit(pd.concat([X_tr, X_va]), pd.concat([y_tr, y_va]))

# 11) Preparar features para el próximo mes
next_dt    = mensual.index.max() + MonthBegin()
last_gmean = gmean.iloc[-1]
last_gstd  = gstd.iloc[-1]

preds = []
for pid in mensual.columns:
    s = mensual[pid]
    r = {"product_id": pid}
    for lag in range(1, MAX_LAG+1):
        r[f"lag_{lag}"] = s.iloc[-lag]
    for w in ROLL:
        win = s.iloc[-w:]
        r[f"mean_{w}"] = win.mean()
        r[f"std_{w}"]  = win.std()
        r[f"min_{w}"]  = win.min()
        r[f"max_{w}"]  = win.max()
    m = next_dt.month - 1
    r["sin_m"] = np.sin(2*np.pi*m/12)
    r["cos_m"] = np.cos(2*np.pi*m/12)
    r["gmean"] = last_gmean
    r["gstd"]  = last_gstd
    preds.append(r)

X_pred = pd.DataFrame(preds)[FEATURES]
y_pred = final_rf.predict(X_pred)
y_pred = np.maximum(0, np.round(y_pred, 5))

# 12) Guardar submission
submission = pd.DataFrame({
    "product_id": mensual.columns,
    "tn":          y_pred
})
submission.to_csv(OUT_SUB, index=False, float_format="%.5f")
print(f"✅ Submission RandomForest reproducible guardada → {OUT_SUB.name}")

[I 2025-07-06 21:55:29,029] A new study created in memory with name: no-name-5443fd40-63ef-48f8-9b31-576d7f1fee68
[I 2025-07-06 21:55:39,006] Trial 0 finished with value: 29.28693022090955 and parameters: {'n_estimators': 600, 'max_depth': 17, 'min_samples_split': 9, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 29.28693022090955.
[I 2025-07-06 21:55:44,400] Trial 1 finished with value: 29.03061168048618 and parameters: {'n_estimators': 400, 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 29.03061168048618.
[I 2025-07-06 21:56:24,022] Trial 2 finished with value: 29.56407724246239 and parameters: {'n_estimators': 700, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 1 with value: 29.03061168048618.
[I 2025-07-06 21:56:26,359] Trial 3 finished with value: 29.23432520273503 and parameters: {'n_estimators': 200, 'max_depth': 9, 'min_samples

✅ Submission RandomForest reproducible guardada → submission_t780_rf_optuna.csv


9- LIGHT GBM MV

In [23]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings("ignore")

# ───────── CONFIG ─────────
BASE       = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
VENTAS     = BASE / "sell-in.txt"
LISTA      = BASE / "780_a_predecir.txt"
OUT_SUB    = BASE / "submission_t780_lgbm_multivar.csv"
SEED       = 14
MAX_LAG    = 12
ROLL       = [3, 6, 12]
VAL_MONTHS = 12
# ──────────────────────────

# 0) Fijo semillas globales
random.seed(SEED)
np.random.seed(SEED)

# 1) Cargo datos y limpio nombres
df = pd.read_csv(VENTAS, sep="\t", engine="python")
df.columns = df.columns.str.strip()
period_col = next(c for c in df.columns if "period" in c.lower())
df = df.rename(columns={period_col: "periodo"})

# 2) Formateo fecha y filtro SKUs
df["periodo"] = df["periodo"].astype(str).str.zfill(6)
df["fecha"]   = pd.to_datetime(
    df["periodo"].str[:4] + df["periodo"].str[4:6] + "01",
    format="%Y%m%d", errors="coerce"
)
with open(LISTA, "r") as f:
    prods = [int(l) for l in f if l.strip() and not l.lower().startswith("product")]
df = df[df["product_id"].isin(prods) & df["fecha"].notna()]

# 3) Agrego por mes/producto las exógenas y el target
agg = df.groupby(["product_id","fecha"]).agg({
    "tn":               "sum",
    "cust_request_qty": "sum",
    "cust_request_tn":  "sum"
}).sort_index()

# 4) Pivot a ancho (cada SKU es columna)
tn_df   = agg["tn"].unstack(0).asfreq("MS", fill_value=0)
rq_df   = agg["cust_request_qty"].unstack(0).asfreq("MS", fill_value=0)
rtn_df  = agg["cust_request_tn"].unstack(0).asfreq("MS", fill_value=0)

# 5) Feature engineering multivariante
records = []
for pid in tn_df.columns:
    s_tn, s_rq, s_rtn = tn_df[pid], rq_df[pid], rtn_df[pid]
    for i in range(MAX_LAG, len(s_tn)):
        fecha = s_tn.index[i]
        row = {"product_id": pid, "fecha": fecha, "target": s_tn.iloc[i]}
        # lags de tn, rq, rtn
        for lag in range(1, MAX_LAG+1):
            row[f"tn_lag_{lag}"]  = s_tn.iloc[i-lag]
            row[f"rq_lag_{lag}"]  = s_rq.iloc[i-lag]
            row[f"rtn_lag_{lag}"] = s_rtn.iloc[i-lag]
        # rolling stats de tn
        for w in ROLL:
            win = s_tn.iloc[i-w:i]
            row[f"tn_mean_{w}"] = win.mean()
            row[f"tn_std_{w}"]  = win.std()
        # rolling exógenas
        for w in ROLL:
            row[f"rq_mean_{w}"]  = s_rq.iloc[i-w:i].mean()
            row[f"rtn_mean_{w}"] = s_rtn.iloc[i-w:i].mean()
        # fecha cíclica
        m = fecha.month - 1
        row["sin_m"] = np.sin(2*np.pi*m/12)
        row["cos_m"] = np.cos(2*np.pi*m/12)
        records.append(row)

df_feat = pd.DataFrame(records).fillna(0)

# 6) Split temporal train/valid
max_date  = tn_df.index.max()
cut_date  = max_date - pd.DateOffset(months=VAL_MONTHS)
train_df  = df_feat[df_feat["fecha"] <= cut_date].copy()
valid_df  = df_feat[df_feat["fecha"] >  cut_date].copy()

# 7) Preparar datos para LightGBM
cat_feats = ["product_id"]
for d in (train_df, valid_df):
    d["product_id"] = d["product_id"].astype("category")

FEATURES   = [c for c in train_df.columns if c not in ["target","fecha"]]
X_train, y_train = train_df[FEATURES], train_df["target"]
X_val,   y_val   = valid_df[FEATURES], valid_df["target"]

# 8) Entreno LGBM con semilla reproducible
model = LGBMRegressor(
    objective="regression",
    metric="rmse",
    random_state=SEED,
    feature_fraction_seed=SEED,
    bagging_seed=SEED,
    learning_rate=0.01,
    num_leaves=64,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_estimators=20_000,
    verbose=-1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    categorical_feature=cat_feats,
    callbacks=[early_stopping(stopping_rounds=100), log_evaluation(period=200)]
)

# 9) Predecir siguiente mes
next_dt   = max_date + MonthBegin()
pred_rows = []
for pid in tn_df.columns:
    s_tn, s_rq, s_rtn = tn_df[pid], rq_df[pid], rtn_df[pid]
    row = {"product_id": pid}
    # lags y rolling igual que antes
    for lag in range(1, MAX_LAG+1):
        row[f"tn_lag_{lag}"]  = s_tn.iloc[-lag]
        row[f"rq_lag_{lag}"]  = s_rq.iloc[-lag]
        row[f"rtn_lag_{lag}"] = s_rtn.iloc[-lag]
    for w in ROLL:
        row[f"tn_mean_{w}"]  = s_tn.iloc[-w:].mean()
        row[f"tn_std_{w}"]   = s_tn.iloc[-w:].std()
        row[f"rq_mean_{w}"]  = s_rq.iloc[-w:].mean()
        row[f"rtn_mean_{w}"] = s_rtn.iloc[-w:].mean()
    m = next_dt.month - 1
    row["sin_m"] = np.sin(2*np.pi*m/12)
    row["cos_m"] = np.cos(2*np.pi*m/12)
    pred_rows.append(row)

pred_df = pd.DataFrame(pred_rows)

# ——— Aquí está la corrección clave ———
# reconstruimos la categoría con las mismas categorías del train
pred_df["product_id"] = pd.Categorical(
    pred_df["product_id"],
    categories=train_df["product_id"].cat.categories
)

X_pred = pred_df[FEATURES]
y_pred = model.predict(X_pred)
y_pred = np.maximum(0, np.round(y_pred, 5))

# 10) Guardar submission
submission = pd.DataFrame({
    "product_id": tn_df.columns,
    "tn":          y_pred
})
submission.to_csv(OUT_SUB, index=False, float_format="%.5f")
print(f"✅ Submission multivar guardada → {OUT_SUB.name}")

Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 34.4833
[400]	valid_0's rmse: 30.6698
Early stopping, best iteration is:
[445]	valid_0's rmse: 30.6057
✅ Submission multivar guardada → submission_t780_lgbm_multivar.csv


10- LSTM

In [5]:
import numpy as np
import pandas as pd
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings("ignore")

# ───────── CONFIG ─────────
BASE       = Path(r"C:/Users/Lenovo/Desktop/LABO 3")
VENTAS     = BASE / "sell-in.txt"
LISTA      = BASE / "780_a_predecir.txt"
OUT_SUB    = BASE / "submission_lstm_multivar.csv"
SEED       = 14
MAX_LAG    = 12
ROLL       = [3, 6, 12]
# ──────────────────────────

# reproducibilidad
tf.random.set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# 1) Cargar datos y lista de productos
df = pd.read_csv(VENTAS, sep="\t", engine="python")
df.columns = df.columns.str.strip()
# detectar columna periodo
date_col = next(c for c in df.columns if "period" in c.lower())
df = df.rename(columns={date_col: "periodo"})
# convertir periodo YYYYMM a fecha primer día
df["periodo"] = df["periodo"].astype(str).str.zfill(6)
df["fecha"] = pd.to_datetime(df["periodo"] + "01", format="%Y%m%d", errors="coerce")
# filtrar lista
with open(LISTA) as f:
    prods = [int(l) for l in f if l.strip().isdigit()]
df = df[df["product_id"].isin(prods) & df["fecha"].notna()]

# 2) Agregar target y exógenas
agg = df.groupby(["product_id","fecha"]).agg({
    "tn":               "sum",
    "cust_request_qty": "sum",
    "cust_request_tn":  "sum"
}).sort_index()

# 3) Pivot wide
tn_df  = agg["tn"].unstack(0).asfreq("MS", fill_value=0)
rq_df  = agg["cust_request_qty"].unstack(0).asfreq("MS", fill_value=0)
rtn_df = agg["cust_request_tn"].unstack(0).asfreq("MS", fill_value=0)

# 4) Feature engineering global
dates = tn_df.index
products = list(tn_df.columns)
# construir matriz multivar
data = pd.concat([tn_df, rq_df, rtn_df], axis=1, keys=["tn","rq","rtn"])
# escala cada serie por producto y exógena
scalers = {}
for key in ["tn","rq","rtn"]:
    for pid in products:
        scaler = MinMaxScaler()
        col = (key, pid)
        data[col] = scaler.fit_transform(data[[col]])
        scalers[col] = scaler

# 5) Crear secuencias (samples, timesteps, features)
feature_cols = [(k,p) for k in ["tn","rq","rtn"] for p in products]
arr = data[feature_cols].values  # shape (T, F)
X, y = [], []
for i in range(MAX_LAG, len(arr)):
    X.append(arr[i-MAX_LAG:i])
    # únicamente target tn para el siguiente mes
    y.append(arr[i, feature_cols.index(("tn", products[0])):feature_cols.index(("tn", products[-1]))+1])
X = np.array(X)
y = np.array(y)

# 6) Split train/val temporal
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

# 7) Definir y entrenar LSTM
model = keras.Sequential([
    keras.layers.LSTM(128, input_shape=(MAX_LAG, X.shape[2]),
                      kernel_initializer=keras.initializers.GlorotUniform(seed=SEED)),
    keras.layers.Dropout(0.2, seed=SEED),
    keras.layers.Dense(len(products), activation='linear')
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100,
          batch_size=32, callbacks=[es], verbose=1)

# 8) Predecir siguiente mes para cada producto
last_seq = arr[-MAX_LAG:]
pred_scaled = model.predict(last_seq.reshape(1, MAX_LAG, -1))[0]
# reconstruir valores tn de cada producto
preds = {}
for i, pid in enumerate(products):
    scaler = scalers[("tn", pid)]
    val = scaler.inverse_transform([[pred_scaled[i]]])[0,0]
    preds[pid] = max(0, val)

# 9) Guardar submission
submission = pd.DataFrame({
    "product_id": products,
    "tn":          [preds[pid] for pid in products]
})
submission.to_csv(OUT_SUB, index=False, float_format="%.5f")
print(f"✅ Submission LSTM multivar guardada → {OUT_SUB.name}")

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - loss: nan - val_loss: nan
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 468ms/step - loss: nan - val_loss: nan
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 403ms/step - loss: nan - val_loss: nan
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step - loss: nan - val_loss: nan
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 336ms/step - loss: nan - val_loss: nan
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step - loss: nan - val_loss: nan
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step - loss: nan - val_loss: nan
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step - loss: nan - val_loss: nan
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step - loss: nan - val_loss: nan
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - loss: nan - val_loss: nan
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step - loss: nan - val_loss: nan
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 884ms/step
✅ Submission LSTM multivar guardada → submission_lstm_multivar.csv
